# Graph Neural Networks — Exercises

Companion to the note [Graph Neural Networks](Graph%20Neural%20Networks.md).

Practise message passing: the GCN propagation rule $H'=\sigma(\tilde D^{-1/2}\tilde A\tilde D^{-1/2}HW)$ by hand and in NumPy, permutation equivariance, GraphSAGE sampling, GAT attention, sum vs mean aggregation (GIN and the 1-WL test), readouts, over-smoothing, and a 2-layer GCN trained with hand-written backpropagation. No deep-learning library needed.

**18 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)

def from_edges(n, edges):
    A = np.zeros((n, n))
    for i, j in edges:
        A[i, j] = A[j, i] = 1
    return A

def sbm(sizes, p_in, p_out, seed=0):
    r = np.random.default_rng(seed)
    labels = np.repeat(np.arange(len(sizes)), sizes)
    Pm = np.where(labels[:, None] == labels[None, :], p_in, p_out)
    U = np.triu(r.random(Pm.shape) < Pm, 1)
    return (U + U.T).astype(float), labels

# Node classification data: 3 communities, features = weak class signal + lots of noise
A_g, y_g = sbm([60, 60, 60], 0.12, 0.01, seed=1)
n_g, n_cls, n_feat = len(y_g), 3, 16
_means = np.random.default_rng(2).normal(size=(n_cls, n_feat))
X_g = 0.5 * _means[y_g] + np.random.default_rng(3).normal(size=(n_g, n_feat))
train_mask = np.zeros(n_g, bool)
train_mask[np.random.default_rng(4).choice(n_g, 36, replace=False)] = True   # 20% labelled
test_mask = ~train_mask

A_path = from_edges(3, [(0, 1), (1, 2)])   # path graph 0 - 1 - 2

## Part A · Concepts

### Exercise 1 · Why AGG must be permutation invariant
*🧠 Concept · ★☆☆*

In $h_v^{(k)}=\mathrm{UPDATE}\big(h_v^{(k-1)}, \mathrm{AGG}(\{h_u^{(k-1)}:u\in N(v)\})\big)$, why must AGG be a function of the **multiset** of neighbour
states (e.g. sum, mean, max) rather than, say, concatenation of the neighbour vectors? Give two reasons. Why does GraphSAGE's LSTM aggregator
need a trick?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Neighbours have no natural order, and different nodes have different numbers of neighbours.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Neighbours have **no canonical order**; concatenation would make the output depend on the arbitrary node numbering, breaking permutation equivariance ([[Graph ML Overview]]).
2. Degrees vary; concatenation would need a fixed input size. Sum/mean/max handle any number of neighbours with the same weights.

An LSTM is order-dependent, so GraphSAGE applies it to a **random permutation** of the (sampled) neighbours, which makes it only approximately invariant.

</details>

### Exercise 2 · GCN vs GraphSAGE vs GAT vs GIN
*🧠 Concept · ★★☆*

For each of the four models in the note's table, state (a) how neighbours are weighted, (b) whether the weights are fixed by the graph or learned,
(c) one typical strength. Which model would you pick for (i) a huge graph that does not fit in GPU memory, (ii) graph classification of molecules where
counting substructures matters, (iii) a citation graph where some neighbours are much more relevant than others?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

GCN's weights are $1/\sqrt{\tilde d_u\tilde d_v}$; GAT computes $\alpha_{vu}$ from the features; GIN uses an unweighted sum.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| Model | Weighting | Fixed/learned | Strength |
|---|---|---|---|
| GCN | $1/\sqrt{\tilde d_u\tilde d_v}$ (symmetric normalisation, self-loop) | fixed by the graph | simple, strong baseline, few parameters |
| GraphSAGE | mean/pool/LSTM over a **sampled** neighbourhood, self concatenated | fixed (mean) | inductive, scalable mini-batching |
| GAT | attention $\alpha_{vu} = \text{softmax}_u(e_{vu})$ | **learned**, feature-dependent | anisotropic: can ignore irrelevant neighbours |
| GIN | sum + MLP, $(1+\epsilon)h_v$ | fixed (sum) | maximally expressive among MPNNs (= 1-WL) |

(i) GraphSAGE (neighbour sampling) or a precomputed SGC. (ii) GIN with sum readout. (iii) GAT.

</details>

### Exercise 3 · Over-smoothing and over-squashing
*🧠 Concept · ★★☆*

(a) What is over-smoothing, and why does it limit GCN depth to typically 2–4 layers? (b) What is over-squashing, and why is it a different problem?
(c) Give two remedies for each.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Over-smoothing: repeated averaging. Over-squashing: exponentially many nodes in the receptive field, one fixed-size vector.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Each GCN layer is a low-pass filter (a weighted average over neighbours). Repeating it drives all node representations in a connected component
towards the same vector (Exercise 10 and 15), so classes become indistinguishable.
(b) With $k$ layers a node's receptive field grows like $d^k$ (Exercise 7); information from far away must be compressed through bottleneck edges into one
fixed-size vector, so long-range dependencies are lost. It is about **too little** information getting through, not representations becoming equal.
(c) Over-smoothing: residual/skip connections and jumping knowledge, initial-residual (GCNII), PairNorm/normalisation, DropEdge, fewer layers.
Over-squashing: graph rewiring (add edges across bottlenecks), virtual/master node, graph transformers with global attention.

</details>

### Exercise 4 · Sum, mean or max?
*🧠 Concept · ★★☆*

Consider neighbour multisets of scalar features. Which aggregators (sum, mean, max) can distinguish
(a) $\{1, 1\}$ vs $\{1\}$, (b) $\{1, 2\}$ vs $\{1, 1, 2, 2\}$, (c) $\{1, 3\}$ vs $\{2, 2\}$, (d) $\{0, 3\}$ vs $\{1, 2, 3\}$?
Why does GIN use **sum** followed by an MLP?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Mean captures the *distribution* of the multiset, max only its *set* (the distinct maximum), sum its full multiplicity-weighted content.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) sum ✔ (2 vs 1), mean ✘, max ✘. (b) sum ✔ (3 vs 6), mean ✘ (1.5 both), max ✘. (c) sum ✘ (4 both), mean ✘, max ✔ (3 vs 2).
(d) sum ✔ (3 vs 6), mean ✔ (1.5 vs 2), max ✘ (3 both).

For scalars no single aggregator wins (c), but Xu et al. (2019) show that **sum over a suitable (MLP-learned) feature map** is injective on bounded-size multisets
of countable features, which mean and max are not. Hence GIN = sum + MLP is as powerful as the 1-WL test ([[Graph ML Overview]]).

</details>

## Part B · By hand

### Exercise 5 · The GCN normalised adjacency
*✍️ By hand · ★☆☆*

For the path graph $0-1-2$ compute $\tilde A = A + I$, the degrees $\tilde d$ and the entries $\hat A_{00}, \hat A_{01}, \hat A_{11}$ of
$\hat A = \tilde D^{-1/2}\tilde A\tilde D^{-1/2}$.

In [ ]:
d_tilde = None
Ahat_00 = None
Ahat_01 = None
Ahat_11 = None

_At = A_path + np.eye(3); _d = _At.sum(1)
_Ah = np.diag(_d ** -0.5) @ _At @ np.diag(_d ** -0.5)
check('degrees with self-loops', d_tilde, _d)
check('Ahat_00', Ahat_00, _Ah[0, 0], tol=1e-4)
check('Ahat_01', Ahat_01, _Ah[0, 1], tol=1e-4)
check('Ahat_11', Ahat_11, _Ah[1, 1], tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$\hat A_{ij} = \tilde A_{ij}/\sqrt{\tilde d_i\tilde d_j}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\tilde d = (2, 3, 2)$. $\hat A_{00} = 1/\sqrt{2\cdot2} = 0.5$, $\hat A_{01} = 1/\sqrt{2\cdot3} = 1/\sqrt6\approx 0.408$, $\hat A_{11} = 1/3$.
The self-loop lets a node keep its own feature; the symmetric normalisation keeps the scale of features stable across layers (eigenvalues of $\hat A$ lie in $(-1, 1]$).

```python
d_tilde = [2, 3, 2]
Ahat_00 = 0.5
Ahat_01 = 1 / np.sqrt(6)
Ahat_11 = 1 / 3
```

</details>

### Exercise 6 · One GCN layer by hand
*✍️ By hand · ★★☆*

On the same path graph, scalar node features $x = (1, 2, 3)$, weight $W = 1$, activation ReLU. Compute $h' = \mathrm{ReLU}(\hat Ax\,W)$.
Interpret $h'_1$ as a weighted average: why is it not simply the mean $2$ of $\{1,2,3\}$?

In [ ]:
h_gcn = None  # [h'_0, h'_1, h'_2]

_At = A_path + np.eye(3); _d = _At.sum(1)
check("h'", h_gcn, np.maximum(0, np.diag(_d ** -0.5) @ _At @ np.diag(_d ** -0.5) @ np.array([1, 2, 3.])), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Row $i$ of $\hat A$ times $x$. Row 1 is $(1/\sqrt6, 1/3, 1/\sqrt6)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$h'_0 = 0.5\cdot1 + \tfrac{1}{\sqrt6}\cdot2 \approx 1.316$, $h'_1 = \tfrac{1}{\sqrt6}(1+3) + \tfrac13\cdot2\approx 2.300$, $h'_2 = \tfrac{2}{\sqrt6} + 0.5\cdot3\approx 2.316$.
The row weights of $\hat A$ do not sum to 1 (row 1 sums to $0.333 + 0.816 = 1.15$): symmetric normalisation is not a plain average. Neighbours with low degree
(nodes 0 and 2, $\tilde d=2$) get *more* weight $1/\sqrt{\tilde d_u\tilde d_v}$ than a mean would give. The random-walk normalisation $\tilde D^{-1}\tilde A$ would give exactly the mean, 2.

```python
h_gcn = [0.5 + 2 / np.sqrt(6), 4 / np.sqrt(6) + 2 / 3, 2 / np.sqrt(6) + 1.5]
```

</details>

### Exercise 7 · Receptive field growth
*✍️ By hand · ★☆☆*

In an infinite tree where every node has degree 4, how many nodes lie within the receptive field (distance $\le k$) of a node after $k = 3$ message-passing layers?
Give the general formula for degree $d$ and $k$ layers. What does this mean for over-squashing and for mini-batch training?

In [ ]:
receptive_3 = None

check('receptive field', receptive_3, sum(4 * 3 ** (j - 1) for j in range(1, 4)) + 1)

<details>
<summary><b>💡 Hint</b></summary>

The root has $d$ children; every other node has $d - 1$ further neighbours.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$1 + d\sum_{j=1}^{k}(d-1)^{j-1} = 1 + 4 + 12 + 36 = 53$. In general $1 + d\frac{(d-1)^k - 1}{d-2}$ for $d>2$, i.e. exponential in $k$.
Over-squashing: 53 (or millions on real graphs) messages must fit into one vector. Mini-batching: computing one node's output needs its whole $k$-hop subgraph ("neighbourhood explosion"), which is why
GraphSAGE samples a fixed number of neighbours per layer (Exercise 16).

```python
receptive_3 = 53
```

</details>

### Exercise 8 · Parameter counts
*✍️ By hand · ★☆☆*

(a) A GCN layer maps 16 input features to 32 outputs, with bias. How many parameters does it have for a graph with 100 nodes? With 1 million nodes?
(b) A single-head GAT layer with the same sizes has a weight matrix $W$ (no bias) and an attention vector $a\in\mathbb R^{2\cdot32}$. How many parameters?

In [ ]:
params_gcn = None
params_gat = None

check('GCN params', params_gcn, 16 * 32 + 32)
check('GAT params', params_gat, 16 * 32 + 2 * 32)

<details>
<summary><b>💡 Hint</b></summary>

The weights are shared across all nodes, like a convolution kernel is shared across all pixels.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $16\cdot32 + 32 = 544$, **independent of the number of nodes** (weight sharing). That is what makes GNNs inductive, unlike a shallow embedding with one vector per node.
(b) $16\cdot 32 + 64 = 576$. $K$ attention heads multiply this by $K$.

```python
params_gcn = 544
params_gat = 576
```

</details>

### Exercise 9 · GAT attention by hand
*✍️ By hand · ★★☆*

Node 0 has neighbours $\{0\ (\text{self}), 1, 2\}$. The raw scores $a^\top[Wh_0\,\|\,Wh_j]$ are $(1, 2, -1)$ for $j = 0, 1, 2$.
GAT applies LeakyReLU (slope 0.2) and a softmax over the neighbourhood. With transformed scalar features $Wh_j = (1, 2, -1)$, compute the attention
weights $\alpha_{0j}$ and the output $h'_0 = \sum_j\alpha_{0j}Wh_j$ (no final nonlinearity).

In [ ]:
alpha_0 = None  # [alpha_00, alpha_01, alpha_02]
h0_gat = None

_e = np.array([1, 2, -1.]); _e = np.where(_e > 0, _e, 0.2 * _e)
_al = np.exp(_e) / np.exp(_e).sum()
check('attention weights', alpha_0, _al, tol=1e-4)
check("h'_0", h0_gat, _al @ np.array([1, 2, -1.]), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

LeakyReLU gives $(1, 2, -0.2)$; then $\alpha_{0j} = e^{e_{0j}}/\sum_k e^{e_{0k}}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$e = (1, 2, -0.2)$, $\exp(e) = (2.718, 7.389, 0.819)$, sum $= 10.926$ → $\alpha_0 \approx (0.249, 0.676, 0.075)$.
$h'_0 \approx 0.249\cdot1 + 0.676\cdot2 - 0.075 \approx 1.526$. Unlike GCN, the weights depend on the features, and neighbour 2 is nearly ignored.
This is attention as in [[Transformers]], but restricted to the graph's edges (a masked attention).

```python
_e = np.array([1, 2, -0.2])
alpha_0 = np.exp(_e) / np.exp(_e).sum()
h0_gat = alpha_0 @ np.array([1, 2, -1.0])
```

</details>

### Exercise 10 · Over-smoothing as a power iteration
*✍️ By hand · ★★★*

Ignore weights and nonlinearities: $k$ GCN layers compute $\hat A^kX$.
(a) Show that $v = \tilde D^{1/2}\mathbf 1$ is an eigenvector of $\hat A$ with eigenvalue 1.
(b) For a connected graph (with self-loops) all other eigenvalues satisfy $|\lambda|<1$. What is $\lim_{k\to\infty}\hat A^kx$?
(c) Compute this limit for the path graph and $x = (1,2,3)$.

In [ ]:
x_limit = None  # limit vector of length 3

_At = A_path + np.eye(3); _d = _At.sum(1); _Ah = np.diag(_d ** -0.5) @ _At @ np.diag(_d ** -0.5)
check('limit of Ahat^k x', x_limit, np.linalg.matrix_power(_Ah, 500) @ np.array([1, 2, 3.]), tol=1e-4)

<details>
<summary><b>💡 Hint 1</b></summary>

$\hat A v = \tilde D^{-1/2}\tilde A\tilde D^{-1/2}\tilde D^{1/2}\mathbf 1 = \tilde D^{-1/2}\tilde A\mathbf 1$, and $\tilde A\mathbf 1 = \tilde d$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Expand $x$ in the orthonormal eigenbasis of the symmetric $\hat A$; every component except the one along $v$ decays like $\lambda^k$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\hat Av = \tilde D^{-1/2}\tilde A\mathbf 1 = \tilde D^{-1/2}\tilde d = \tilde D^{1/2}\mathbf 1 = v$ ✔.
(b) $\hat A^kx\to\frac{v v^\top x}{v^\top v}$: every node's feature becomes proportional to $\sqrt{\tilde d_i}$ times one shared number, so **the feature carries no information about the node except its degree** (over-smoothing).
(c) $v = (\sqrt2,\sqrt3,\sqrt2)$, $v^\top v = 7$, $v^\top x = \sqrt2 + 2\sqrt3 + 3\sqrt2 = 4\sqrt2 + 2\sqrt3\approx 9.121$.
Limit $= v\cdot 9.121/7 \approx (1.843, 2.257, 1.843)$. Nodes 0 and 2, which started at 1 and 3, have become identical.

```python
_v = np.sqrt([2, 3, 2.])
x_limit = _v * (_v @ np.array([1, 2, 3.])) / (_v @ _v)
```

</details>

## Part C · Message passing in NumPy

### Exercise 11 · Normalised adjacency in NumPy
*💻 Code · ★☆☆*

Implement `normalize_adj(A)` returning $\hat A = \tilde D^{-1/2}(A+I)\tilde D^{-1/2}$ without forming a diagonal matrix (use broadcasting).
Verify that $\hat A$ is symmetric and its largest eigenvalue is 1 for `A_g`.

In [ ]:
def normalize_adj(A):
    # TODO
    return None

Ahat_g = normalize_adj(A_g)

_At = A_g + np.eye(n_g); _Dm = np.diag(1 / np.sqrt(_At.sum(1)))
check('normalized adjacency', Ahat_g, _Dm @ _At @ _Dm)
check('largest eigenvalue is 1', None if Ahat_g is None else np.linalg.eigvalsh(Ahat_g).max(), 1.0)

<details>
<summary><b>💡 Hint</b></summary>

`s = 1 / np.sqrt(At.sum(1))`; then `s[:, None] * At * s[None, :]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Broadcasting costs $O(n^2)$ instead of two $O(n^3)$ dense matrix products; with `scipy.sparse` it is $O(|E|)$, which is what PyG/DGL do.

```python
def normalize_adj(A):
    At = A + np.eye(len(A))
    s = 1 / np.sqrt(At.sum(1))
    return s[:, None] * At * s[None, :]

Ahat_g = normalize_adj(A_g)
```

</details>

### Exercise 12 · GCN layer and permutation equivariance
*💻 Code · ★★☆*

Implement `gcn_layer(Ahat, H, W, b, act=True)` $= \mathrm{ReLU}(\hat AHW + b)$ (no ReLU if `act=False`). Then verify **equivariance**: for a random
permutation matrix $P$, `gcn_layer(P Ahat P^T, P H, W, b)` must equal `P @ gcn_layer(Ahat, H, W, b)`.
Store the maximum absolute difference in `equiv_err`.

In [ ]:
def gcn_layer(Ahat, H, W, b, act=True):
    # TODO
    return None

W_t, b_t = rng.normal(size=(n_feat, 8)), rng.normal(size=8)
Perm = np.eye(n_g)[rng.permutation(n_g)]
equiv_err = None

# reference: explicit per-node message passing with the GCN weights 1/sqrt(d_u d_v)
_dt = A_g.sum(1) + 1
_ref = np.array([sum(X_g[u] / np.sqrt(_dt[u] * _dt[v]) for u in list(np.flatnonzero(A_g[v])) + [v]) for v in range(n_g)]) @ W_t + b_t
_got = None if Ahat_g is None or gcn_layer(Ahat_g, X_g, W_t, b_t) is None else gcn_layer(Ahat_g, X_g, W_t, b_t, act=False)
check('GCN layer = explicit message passing', _got, _ref)
check('equivariance error ~ 0', equiv_err, 0.0, tol=1e-10)

<details>
<summary><b>💡 Hint</b></summary>

Equivariance check: `np.abs(gcn_layer(Perm @ Ahat_g @ Perm.T, Perm @ X_g, W_t, b_t) - Perm @ gcn_layer(Ahat_g, X_g, W_t, b_t)).max()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(P\hat AP^\top)(PH)W = P\hat A(P^\top P)HW = P\hat AHW$, and ReLU and the bias act row-wise, so they commute with $P$. The explicit loop shows that the matrix form
is just "sum over $N(v)\cup\{v\}$ of $h_u/\sqrt{\tilde d_u\tilde d_v}$, then a shared linear map".

```python
def gcn_layer(Ahat, H, W, b, act=True):
    Z = Ahat @ H @ W + b
    return np.maximum(Z, 0) if act else Z

W_t, b_t = rng.normal(size=(n_feat, 8)), rng.normal(size=8)
Perm = np.eye(n_g)[rng.permutation(n_g)]
equiv_err = np.abs(gcn_layer(Perm @ Ahat_g @ Perm.T, Perm @ X_g, W_t, b_t) - Perm @ gcn_layer(Ahat_g, X_g, W_t, b_t)).max()
```

</details>

### Exercise 13 · SGC: propagate, then classify
*💻 Code · ★★☆*

Simplified Graph Convolution (Wu et al., 2019) drops the nonlinearities: features $\hat A^KX$ followed by logistic regression.
For $K = 0, 1, 2, 3, 5, 10, 30$ fit `LogisticRegression(max_iter=2000)` on the training nodes of `X_g` and report test accuracy in `acc_sgc` (dict).
What happens for $K=0$ (no graph) and for very large $K$?

In [ ]:
acc_sgc = None  # {K: accuracy}

if acc_sgc is not None:
    check('graph propagation (K=2) beats no graph (K=0) by > 0.15', acc_sgc[2] > acc_sgc[0] + 0.15, True)
else:
    check('graph propagation (K=2) beats no graph (K=0) by > 0.15', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Precompute `F = X_g.copy()`; then repeatedly `F = Ahat_g @ F` and fit on `F[train_mask]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$K=0$ is a plain linear classifier on noisy features (≈ 0.65 here). Propagating over 1–3 hops averages the noise away within communities (homophily), and accuracy
jumps to ≈ 0.94 ($K=1$) and ≈ 0.97–0.98 ($K=2$–$5$). For large $K$ the features over-smooth (Exercise 10): ≈ 0.82 at $K=10$ and ≈ 0.33, i.e. chance level, at $K=30$. SGC is very cheap: propagation is a preprocessing step.

```python
acc_sgc, F = {}, X_g.copy()
for K in range(31):
    if K in (0, 1, 2, 3, 5, 10, 30):
        clf = LogisticRegression(max_iter=2000).fit(F[train_mask], y_g[train_mask])
        acc_sgc[K] = clf.score(F[test_mask], y_g[test_mask])
    F = Ahat_g @ F
print({k: round(v, 3) for k, v in acc_sgc.items()})
```

</details>

### Exercise 14 · A 2-layer GCN with hand-written backprop
*💻 Code · ★★★*

Implement `gcn_loss_grads(Ahat, X, y, mask, params)` for
$$Z_1 = \hat AXW_1 + b_1,\quad H_1 = \mathrm{ReLU}(Z_1),\quad Z_2 = \hat AH_1W_2 + b_2,\quad P = \mathrm{softmax}(Z_2),$$
with loss = mean cross-entropy over nodes in `mask`. Return `(loss, grads)` where `grads` has the same keys as `params` (`'W1','b1','W2','b2'`).
Train with full-batch gradient descent (lr 0.2, 300 epochs, hidden size 16) on the training nodes and store the test accuracy in `acc_gcn`.

In [ ]:
def gcn_loss_grads(Ahat, X, y, mask, params):
    # TODO: forward pass, then backward pass
    return None, None

def init_params(d_in, d_hid, d_out, seed=0):
    r = np.random.default_rng(seed)
    return {'W1': r.normal(0, np.sqrt(2 / d_in), (d_in, d_hid)), 'b1': np.zeros(d_hid),
            'W2': r.normal(0, np.sqrt(2 / d_hid), (d_hid, d_out)), 'b2': np.zeros(d_out)}

acc_gcn = None

# finite-difference gradient check on a tiny random problem
_r = np.random.default_rng(7)
_A = (np.triu(_r.random((6, 6)) < 0.5, 1)).astype(float); _A = _A + _A.T
_Ah = (_A + np.eye(6)) / np.sqrt(np.outer(_A.sum(1) + 1, _A.sum(1) + 1))
_X, _y, _m = _r.normal(size=(6, 4)), _r.integers(0, 3, 6), np.array([1, 1, 0, 1, 0, 1], bool)
_p = init_params(4, 5, 3, seed=1)
_loss, _g = gcn_loss_grads(_Ah, _X, _y, _m, _p)
if _g is not None:
    _num = {}
    for k in _p:
        _num[k] = np.zeros_like(_p[k])
        for idx in np.ndindex(_p[k].shape):
            _q = {kk: vv.copy() for kk, vv in _p.items()}; _q[k][idx] += 1e-6
            _num[k][idx] = (gcn_loss_grads(_Ah, _X, _y, _m, _q)[0] - _loss) / 1e-6
    check('gradients match finite differences', np.concatenate([_g[k].ravel() for k in _p]), np.concatenate([_num[k].ravel() for k in _p]), tol=1e-4)
else:
    check('gradients match finite differences', None, 0)
check('test accuracy > 0.85', None if acc_gcn is None else acc_gcn > 0.85, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Backward: `dZ2 = (P - Y) / n_train` on masked rows and 0 elsewhere; `dW2 = (Ahat @ H1).T @ dZ2`; `db2 = dZ2.sum(0)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`dH1 = Ahat.T @ dZ2 @ W2.T`; `dZ1 = dH1 * (Z1 > 0)`; `dW1 = (Ahat @ X).T @ dZ1`; `db1 = dZ1.sum(0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Only the training nodes contribute to the loss, but **all** nodes' features flow through $\hat A$, so unlabelled neighbours still shape the representation:
this is semi-supervised learning on graphs (Kipf & Welling, 2017). With 36 labelled nodes the GCN reaches ≈ 1.0 test accuracy here, far above a feature-only classifier.

```python
def gcn_loss_grads(Ahat, X, y, mask, params):
    W1, b1, W2, b2 = params['W1'], params['b1'], params['W2'], params['b2']
    AX = Ahat @ X
    Z1 = AX @ W1 + b1
    H1 = np.maximum(Z1, 0)
    AH = Ahat @ H1
    Z2 = AH @ W2 + b2
    Z2s = Z2 - Z2.max(1, keepdims=True)
    P = np.exp(Z2s) / np.exp(Z2s).sum(1, keepdims=True)
    Y = np.eye(Z2.shape[1])[y]
    n_tr = mask.sum()
    loss = -np.sum(np.log(P[mask][np.arange(n_tr), y[mask]])) / n_tr
    dZ2 = np.where(mask[:, None], (P - Y) / n_tr, 0.0)
    dW2 = AH.T @ dZ2; db2 = dZ2.sum(0)
    dH1 = Ahat.T @ dZ2 @ W2.T
    dZ1 = dH1 * (Z1 > 0)
    dW1 = AX.T @ dZ1; db1 = dZ1.sum(0)
    return loss, {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

def init_params(d_in, d_hid, d_out, seed=0):
    r = np.random.default_rng(seed)
    return {'W1': r.normal(0, np.sqrt(2 / d_in), (d_in, d_hid)), 'b1': np.zeros(d_hid),
            'W2': r.normal(0, np.sqrt(2 / d_hid), (d_hid, d_out)), 'b2': np.zeros(d_out)}

params = init_params(n_feat, 16, n_cls)
for epoch in range(300):
    loss, grads = gcn_loss_grads(Ahat_g, X_g, y_g, train_mask, params)
    for k in params:
        params[k] -= 0.2 * grads[k]
H1 = np.maximum(Ahat_g @ X_g @ params['W1'] + params['b1'], 0)
pred = (Ahat_g @ H1 @ params['W2'] + params['b2']).argmax(1)
acc_gcn = np.mean(pred[test_mask] == y_g[test_mask])
print('final train loss', round(loss, 4), ' test accuracy', round(acc_gcn, 3))
```

</details>

### Exercise 15 · Watching over-smoothing happen
*💻 Code · ★★☆*

Track the normalised **Dirichlet energy** $E(F) = \frac{\operatorname{tr}(F^\top LF)}{\lVert F\rVert_F^2}$ (with $L = I-\hat A$, the normalised Laplacian
with self-loops) of $F = \hat A^kX_g$ for $k = 0,\dots,64$. Store the energies in the array `energy` (length 65). How fast does it decay?

In [ ]:
energy = None

if energy is not None:
    check('energy decreases monotonically', np.all(np.diff(energy) <= 1e-12), True)
    check('energy after 64 layers < 2% of the start', energy[64] < 0.02 * energy[0], True)
else:
    check('energy decreases monotonically', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`L = np.eye(n_g) - Ahat_g`; `np.trace(F.T @ L @ F) / np.sum(F**2)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$E(F) = \frac12\sum_{ij}\hat A_{ij}\lVert f_i/\sqrt{\tilde d_i} - f_j/\sqrt{\tilde d_j}\rVert^2/\lVert F\rVert^2$ measures how different neighbouring (degree-rescaled) features are.
Each multiplication by $\hat A$ shrinks the non-leading eigencomponents by $|\lambda_i|<1$, so the energy decays roughly geometrically (fast within the
first few layers, then at the rate of the second-largest $|\lambda|$). Once $E\approx0$ the features are just $\sqrt{\tilde d_i}$ times a shared vector (Exercise 10).

```python
L_norm = np.eye(n_g) - Ahat_g
F, energy = X_g.copy(), []
for k in range(65):
    energy.append(np.trace(F.T @ L_norm @ F) / np.sum(F ** 2))
    F = Ahat_g @ F
energy = np.array(energy)
print(np.round(energy[[0, 1, 2, 4, 8, 16, 32, 64]], 4))
```

</details>

### Exercise 16 · GraphSAGE with neighbour sampling
*💻 Code · ★★☆*

Implement `sage_layer(A, H, W_self, W_neigh, n_samples, seed)` computing
$h'_v = \mathrm{ReLU}\big(W_\text{self}^\top h_v + W_\text{neigh}^\top\,\mathrm{mean}_{u\in S(v)}h_u\big)$ followed by $\ell_2$-normalisation of each row,
where $S(v)$ is a uniform sample (without replacement) of at most `n_samples` neighbours (isolated nodes use a zero vector).
Check that with `n_samples >= max degree` it equals the full mean aggregator.

In [ ]:
def sage_layer(A, H, W_self, W_neigh, n_samples, seed=0):
    # TODO
    return None

Ws, Wn = rng.normal(size=(n_feat, 8)), rng.normal(size=(n_feat, 8))
H_sampled = sage_layer(A_g, X_g, Ws, Wn, n_samples=3)

_deg = A_g.sum(1, keepdims=True)
_mean = np.divide(A_g @ X_g, _deg, out=np.zeros_like(X_g), where=_deg > 0)
_Z = np.maximum(X_g @ Ws + _mean @ Wn, 0)
_ref = _Z / np.maximum(np.linalg.norm(_Z, axis=1, keepdims=True), 1e-12)
check('full-neighbourhood SAGE = mean aggregator', sage_layer(A_g, X_g, Ws, Wn, n_samples=int(A_g.sum(1).max())), _ref)

<details>
<summary><b>💡 Hint</b></summary>

`nb = np.flatnonzero(A[v])`; `S = r.choice(nb, min(n_samples, len(nb)), replace=False)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Sampling caps the cost per node at $\prod_\ell s_\ell$ (e.g. 25·10 for two layers) regardless of degree, which makes mini-batch training on huge graphs possible.
The sampled mean is an unbiased estimate of the full mean (its variance shrinks as `n_samples` grows). Concatenating/adding the self term separately (instead of a self-loop)
lets the model weigh a node's own state differently from its neighbours'.

```python
def sage_layer(A, H, W_self, W_neigh, n_samples, seed=0):
    r = np.random.default_rng(seed)
    agg = np.zeros_like(H)
    for v in range(len(A)):
        nb = np.flatnonzero(A[v])
        if len(nb):
            S = r.choice(nb, min(n_samples, len(nb)), replace=False)
            agg[v] = H[S].mean(0)
    Z = np.maximum(H @ W_self + agg @ W_neigh, 0)
    return Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-12)

Ws, Wn = rng.normal(size=(n_feat, 8)), rng.normal(size=(n_feat, 8))
H_sampled = sage_layer(A_g, X_g, Ws, Wn, n_samples=3)
```

</details>

### Exercise 17 · A GAT layer
*💻 Code · ★★★*

Implement a single-head `gat_layer(A, H, W, a, slope=0.2)`:
$e_{vu} = \mathrm{LeakyReLU}\big(a^\top[Wh_v\,\|\,Wh_u]\big)$ for $u\in N(v)\cup\{v\}$, $\alpha_{vu} = \mathrm{softmax}_u(e_{vu})$, $h'_v = \sum_u\alpha_{vu}Wh_u$.
Return `(H_out, alpha)` with `alpha` an $n\times n$ matrix (zero for non-neighbours). Vectorise it: split $a = (a_1, a_2)$ so that $a^\top[x\|y] = a_1^\top x + a_2^\top y$.

In [ ]:
def gat_layer(A, H, W, a, slope=0.2):
    # TODO
    return None, None

W_att, a_att = rng.normal(size=(n_feat, 8)) / 4, rng.normal(size=16) / 4
H_gat, alpha_gat = gat_layer(A_g, X_g, W_att, a_att)

if alpha_gat is not None:
    _mask = (A_g + np.eye(n_g)) > 0
    check('rows of alpha sum to 1', alpha_gat.sum(1), np.ones(n_g))
    check('alpha is zero off the graph', np.abs(alpha_gat[~_mask]).max(), 0.0)
    # brute-force reference for node 5
    _WH = X_g @ W_att; _nb = np.flatnonzero(_mask[5])
    _e = np.array([np.r_[_WH[5], _WH[u]] @ a_att for u in _nb]); _e = np.where(_e > 0, _e, 0.2 * _e)
    _al = np.exp(_e) / np.exp(_e).sum()
    check('output of node 5', H_gat[5], _al @ _WH[_nb])
else:
    check('GAT layer', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

`WH = H @ W`; `s = (WH @ a[:d])[:, None] + (WH @ a[d:])[None, :]` gives all pairwise scores at once.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Mask non-edges with `-np.inf` before the row-wise softmax (subtract the row max for stability).

</details>

<details>
<summary><b>✅ Solution</b></summary>

The vectorised form computes all $n^2$ scores, then masks; real implementations compute scores only for the $|E|$ edges (gather/scatter).
Multi-head GAT runs $K$ such layers in parallel and concatenates (hidden layers) or averages (output layer) the results.
Because $\alpha$ depends on both endpoints, GAT can learn to down-weight heterophilous neighbours, something GCN's fixed weights cannot do.

```python
def gat_layer(A, H, W, a, slope=0.2):
    WH = H @ W; d = WH.shape[1]
    s = (WH @ a[:d])[:, None] + (WH @ a[d:])[None, :]
    s = np.where(s > 0, s, slope * s)
    s = np.where((A + np.eye(len(A))) > 0, s, -np.inf)
    s = s - s.max(1, keepdims=True)
    alpha = np.exp(s); alpha /= alpha.sum(1, keepdims=True)
    return alpha @ WH, alpha

W_att, a_att = rng.normal(size=(n_feat, 8)) / 4, rng.normal(size=16) / 4
H_gat, alpha_gat = gat_layer(A_g, X_g, W_att, a_att)
```

</details>

### Exercise 18 · Readout: sum vs mean for graph classification
*💻 Code · ★★☆*

Graph-level tasks need a **readout** $h_G = \mathrm{READOUT}(\{h_v^{(K)}\})$. Take star graphs with 3 and with 6 leaves, all node features equal to 1.
Run two layers of (a) **mean** aggregation with mean readout, (b) **sum** aggregation (GIN-style, $h_v' = h_v + \sum_{u\in N(v)}h_u$, no MLP) with sum readout.
Store the four graph embeddings (scalars) in `readouts = {'mean': (star3, star6), 'sum': (star3, star6)}`. Which combination can tell the graphs apart?

In [ ]:
def star(k):
    return from_edges(k + 1, [(0, i) for i in range(1, k + 1)])

readouts = None

if readouts is not None:
    check('mean/mean cannot distinguish', readouts['mean'][0] == readouts['mean'][1], True)
    check('sum/sum distinguishes', readouts['sum'][0] != readouts['sum'][1], True)
else:
    check('readouts', None, True)

<details>
<summary><b>💡 Hint</b></summary>

With constant features, a mean over any non-empty multiset of 1s is 1. Sum aggregation counts neighbours.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Mean/mean gives 1 for both stars: mean aggregation only sees the *distribution* of neighbour features, not their count, so with uninformative features it cannot even
compute degrees. Sum: layer 1 gives the centre $1 + k$ and leaves 2; layer 2 gives the centre $(1+k) + 2k$ and each leaf $2 + (1+k)$; the sum readout is
$(1+3k) + k(3+k)$, i.e. **28 for $k=3$ and 73 for $k=6$**. This is why GIN uses sum aggregation *and* sum readout for graph classification (e.g. molecules, where counts matter).

```python
def star(k):
    return from_edges(k + 1, [(0, i) for i in range(1, k + 1)])

def embed(A, agg, layers=2):
    h = np.ones(len(A))
    for _ in range(layers):
        h = A @ h / A.sum(1) if agg == 'mean' else h + A @ h
    return h.mean() if agg == 'mean' else h.sum()

readouts = {agg: (embed(star(3), agg), embed(star(6), agg)) for agg in ('mean', 'sum')}
print(readouts)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Graph Neural Networks](Graph%20Neural%20Networks.md).*